# **VALIDATION RULES**

## Main


### Load data

In [ ]:
from load_data import load_input_data

# Load data
df_pipes, df_cctv, df_defects, df_hydraulics = load_input_data(
    source_type="database", # "database" or "excel"
    source_path="sewer_database_example.db", # Path to the input data file
    sheet_names= None # Required only for Excel input. Example: ["PIPES", "CCTV", "DEFECTS", "HYDRAULIC_PROPERTIES"]
    )

### Select output path

In [ ]:
# Select output path
output_path = "Validated_data.xlsx"

## Preprocessing

In [ ]:
from data_preprocessing import drop_all_nan_columns, filter_valid_defects, remove_uncompleted_without_defects

In [ ]:
# Removes columns from database that are not being used
df_pipes, df_hydraulics, df_cctv, df_defects = drop_all_nan_columns(df_pipes, df_hydraulics, df_cctv, df_defects)

In [ ]:
# Keeps only defects and removes other observations (features)

# Dictionary (1 = valid defect, 0 = non-defect / feature)
def_or_fea_map = {
    "IS": 0, "WL": 0, "LO": 0, "DG": 1, "GP": 0, "LB": 0, "OT": 1,
    "IA": 0, "MHJ": 1, "JF": 1, "ED": 1, "GC": 0, "IE": 0, "L": 0,
    "JD": 1, "RI": 1, "SD": 1, "LF": 1, "MC": 0, "CF": 0, "CC": 1,
    "LP": 1, "LD": 0, "IP": 1, "DP": 1, "PH": 1, "JO": 1, "CL": 1,
    "OP": 1, "DE": 1, "LX": 1, "CM": 1, "SV": 1, "PF": 1, "PB": 1,
    "TM": 1, "DF": 1, "DC": 0, "LOV": 0, "PR": 0, "PL": 1, "B": 1, "EX": 1
}

df_defects = filter_valid_defects(df_defects,def_or_fea_map)

In [ ]:
df_cctv = remove_uncompleted_without_defects(df_cctv, df_defects)

## Validation rules of pipes

In [ ]:
from count_changes import evaluate_rule_change

from validation_rules import remove_nan_pipe_id, remove_duplicate_pipe_id, clean_material_column,filter_installation_year, reclassify_sewer_category, replace_zero_diameter_with_nan, set_diameter_outliers_to_nan, set_small_diameters_to_nan, set_small_pipe_length_to_nan, set_negative_slope_to_nan,  set_invalid_invert_difference_to_nan, set_high_slope_to_nan, set_negative_depths_to_nan, set_excessive_depth_to_nan, set_non_positive_flows_to_nan

### Pipes ID

In [ ]:
# Removes pipes without ID
df_pipes = remove_nan_pipe_id(df_pipes)

In [ ]:
# Removes duplicate pipes
df_pipes = remove_duplicate_pipe_id(df_pipes)

### Material

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Converts to NaN
df_pipes = clean_material_column(df_pipes)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Material',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

### Installation year

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Dictionary with Installation year valid ranges according to the pipe material
material_year_limits = {
    "VC":  {"min": 1880},
    "AC":  {"min": 1920, "max": 2016},
    "GRP": {"min": 1970},
    "PE":  {"min": 1985},
    "CONC":{"min": 1910},
    "PVC": {"min": 1950},
    "CI":  {"min": 1880},
    "DI":  {"min": 1955},
    "FRP": {"min": 1970}
}

# Convert to NaN installation years out of valid ranges
df_pipes["Installation_year"] = df_pipes.apply(
    filter_installation_year,
    axis=1,
    limits_dict=material_year_limits
)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Installation_year',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

### Sewer category

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform transmission pipes to local when the diameter is less than or equal to a threshold diameter
df_pipes = reclassify_sewer_category(df_pipes, threshold_diameter=100)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Sewer_category',
    df_cctv=df_cctv,
    change_type='value_change'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

### Diameter

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform diameters equal to 0 to NaN
df_pipes = replace_zero_diameter_with_nan(df_pipes)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# Convert to NaN diameters exceeding a reasonable upper limit according to the material

# Dictionary with max. diameters
diameter_outlier_max = {
    "AC": 600,
    "CONC": 3600,
    "VC": 675,
    "PVC": 1500,
    "PE": 3200
}

# Apply validation rule
df_pipes = set_diameter_outliers_to_nan(df_pipes, diameter_outlier_max)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform diameters smaller than a minimum diameter to NaN
df_pipes = set_small_diameters_to_nan(df_pipes, min_diameter=100)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Diameter',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

### Length

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform lengths that are less than a minimum length to NaN
df_pipes = set_small_pipe_length_to_nan(df_pipes, min_length=0.6)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Pipe_length',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

### Slope

In [ ]:
# Calculate slope

# Calculate the difference between invert levels
df_pipes["Dif_invert"] = df_pipes["UP_depth"] - df_pipes["DW_depth"]

# Calculate the slope with the
df_pipes['Slope'] = (df_pipes["Dif_invert"]/df_pipes["Pipe_length"])*100

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# When the slope is negative or equal to zero, transform slopes, invert levels, and depths to NaN
df_pipes = set_negative_slope_to_nan(df_pipes)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

# When the difference between the inverts is greater than the length of the pipe, transform slopes, invert levels, and depths to NaN
df_pipes = set_invalid_invert_difference_to_nan(df_pipes)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#When the slope is greater than a maximum value, transform slopes, invert levels, and depths to NaN
df_pipes = set_high_slope_to_nan(df_pipes, max_slope=30)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Slope',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Drop temporary column Dif_invert
df_pipes = df_pipes.drop("Dif_invert", axis=1)

### Depth

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#Transform invert levels and depths to NaN when the depth is a negative value
df_pipes = set_negative_depths_to_nan(df_pipes)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Depth',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_pipes.copy()

#When the average depth is greater than a maximum value, transform slopes, invert levels, and depths to NaN
df_pipes = set_excessive_depth_to_nan(df_pipes, max_depth=15)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_pipes,
    column='Depth',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

## Validation rules of Hydraulic Properties

### Flow rate

In [ ]:
# Copy of DataFrame before applying the rule for change tracking
df_before = df_hydraulics.copy()

# Convert to NaN the flow rates that are <= 0 (dry and wet weather)
df_hydraulics = set_non_positive_flows_to_nan(df_hydraulics)

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_hydraulics,
    column='Dry_peak_flow_rate',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

In [ ]:
# Count changes
total, with_cctv = evaluate_rule_change(
    df_before=df_before,
    df_after=df_hydraulics,
    column='Wet_peak_flow_rate',
    df_cctv=df_cctv,
    change_type='to_nan'
)

print(f"Total changed: {total}")
print(f"Changed with CCTV: {with_cctv}")

 # **DOWNLOAD DATA**

In [ ]:
from export_validated_data import export_dataframes_to_excel
export_dataframes_to_excel(
    output_path=output_path,
    df_pipes=df_pipes,
    df_hydraulics=df_hydraulics,
    df_cctv=df_cctv,
    df_defects=df_defects
)